# Linear Regression Practice
**Dataset:** Healthcare  
**Target:** `Billing Amount`  
**Guide:** Follows the full data science pipeline — exploration → preprocessing → tuning → evaluation

---

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import PowerTransformer

import warnings
warnings.filterwarnings('ignore')

## 2. Read Dataset
Read the dataset and preview data summary — equivalent to `skim_without_charts()` in R.

In [2]:
df = pd.read_csv("/Users/tytatee/Library/CloudStorage/GoogleDrive-possakorn.k@gmail.com/My Drive/00_script/c_github/data-scientist-sandbox/ds_sandbox/ds/healthcare_dataset.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (10000, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Tiffany Ramirez,81,Female,O-,Diabetes,2022-11-17,Patrick Parker,Wallace-Hamilton,Medicare,37490.983364,146,Elective,2022-12-01,Aspirin,Inconclusive
1,Ruben Burns,35,Male,O+,Asthma,2023-06-01,Diane Jackson,"Burke, Griffin and Cooper",UnitedHealthcare,47304.064845,404,Emergency,2023-06-15,Lipitor,Normal
2,Chad Byrd,61,Male,B-,Obesity,2019-01-09,Paul Baker,Walton LLC,Medicare,36874.896997,292,Emergency,2019-02-08,Lipitor,Normal
3,Antonio Frederick,49,Male,B-,Asthma,2020-05-02,Brian Chandler,Garcia Ltd,Medicare,23303.322092,480,Urgent,2020-05-03,Penicillin,Abnormal
4,Mrs. Brandy Flowers,51,Male,O-,Arthritis,2021-07-09,Dustin Griffin,"Jones, Brown and Murray",UnitedHealthcare,18086.344184,477,Urgent,2021-08-02,Paracetamol,Normal


In [3]:
# Data summary — equivalent to skim_without_charts()
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Numeric Summary ===")
df.describe()

=== Data Types ===
Name                   object
Age                     int64
Gender                 object
Blood Type             object
Medical Condition      object
Date of Admission      object
Doctor                 object
Hospital               object
Insurance Provider     object
Billing Amount        float64
Room Number             int64
Admission Type         object
Discharge Date         object
Medication             object
Test Results           object
dtype: object

=== Missing Values ===
Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

=== Numeric Summary ===


,Age,Billing Amount,Room Number
count,10000.000000,10000.000000,10000.000000
mean,51.452200,25516.806778,300.082000
std,19.588974,14067.292709,115.806027
min,18.000000,1000.180837,101.000000
25%,35.000000,13506.523967,199.000000
50%,52.000000,25258.112566,299.000000
75%,68.000000,37733.913727,400.000000
max,85.000000,49995.902283,500.000000


In [4]:
# Categorical columns summary
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")
for col in cat_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head(5))

Categorical columns: ['Name', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']

Name: 9378 unique values
Name
Michael Johnson      7
James Johnson        6
Michael Miller       4
Michelle Williams    4
Scott Smith          4
Name: count, dtype: int64

Gender: 2 unique values
Gender
Female    5075
Male      4925
Name: count, dtype: int64

Blood Type: 8 unique values
Blood Type
AB-    1275
AB+    1258
B-     1252
O+     1248
O-     1244
Name: count, dtype: int64

Medical Condition: 6 unique values
Medical Condition
Asthma          1708
Cancer          1703
Hypertension    1688
Arthritis       1650
Obesity         1628
Name: count, dtype: int64

Date of Admission: 1815 unique values
Date of Admission
2019-04-12    15
2022-04-27    15
2021-10-23    14
2023-03-27    14
2022-10-01    14
Name: count, dtype: int64

Doctor: 9416 unique values
Doctor
Michael Johnson     

## 3. Exploratory Data Analysis

In [5]:
# Distribution of target variable: Billing Amount
fig = px.histogram(df, x="Billing Amount", nbins=50, title="Distribution of Billing Amount")
fig.show()

In [6]:
# Count by categorical variables
cat_plot_cols = ["Gender", "Medical Condition", "Admission Type", "Insurance Provider"]
fig = make_subplots(rows=2, cols=2, subplot_titles=cat_plot_cols)

for i, col in enumerate(cat_plot_cols):
    counts = df[col].value_counts()
    row, col_pos = divmod(i, 2)
    fig.add_trace(
        go.Bar(x=counts.index, y=counts.values, name=col),
        row=row + 1, col=col_pos + 1
    )

fig.update_layout(height=500, title="Count by Categorical Variables", showlegend=False)
fig.show()

In [7]:
# Billing Amount by Medical Condition
fig = px.box(df, x="Medical Condition", y="Billing Amount",
             title="Billing Amount by Medical Condition", color="Medical Condition")
fig.show()

## 4. Output Manipulation
Create `log_billing` as the response variable (log-transform stabilises the skewed target),  
then convert date columns and engineer `length_of_stay` — equivalent to `mutate(log_price = log(price))`.

In [8]:
# Convert dates and engineer length_of_stay
df["Date of Admission"] = pd.to_datetime(df["Date of Admission"])
df["Discharge Date"]    = pd.to_datetime(df["Discharge Date"])
df["length_of_stay"]    = (df["Discharge Date"] - df["Date of Admission"]).dt.days

# Create log-transformed target
df["log_billing"] = np.log(df["Billing Amount"])

print(df[["Billing Amount", "log_billing", "length_of_stay"]].describe())

       Billing Amount   log_billing  length_of_stay
count    10000.000000  10000.000000    10000.000000
mean     25516.806778      9.905336       15.561800
std      14067.292709      0.815778        8.612038
min       1000.180837      6.907936        1.000000
25%      13506.523967      9.510928        8.000000
50%      25258.112566     10.136903       16.000000
75%      37733.913727     10.538315       23.000000
max      49995.902283     10.819696       30.000000


In [9]:
# Compare original vs log-transformed target
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Billing Amount (original)", "log(Billing Amount)"])
fig.add_trace(go.Histogram(x=df["Billing Amount"], nbinsx=50, name="Original"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["log_billing"],    nbinsx=50, name="Log"),      row=1, col=2)
fig.update_layout(height=400, title="Target Distribution", showlegend=False)
fig.show()

## 5. Data Splitting
Split into train/test (75/25) and prepare 10-fold cross-validation —  
equivalent to `initial_split()` + `vfold_cv(v=10)` in R tidymodels.

In [10]:
# Define features and target
feature_cols = ["Age", "Gender", "Blood Type", "Medical Condition",
                "Insurance Provider", "Admission Type", "Medication",
                "Test Results", "length_of_stay"]
target_col = "log_billing"

X = df[feature_cols]
y = df[target_col]

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=1311)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows | Total: {X.shape[0]} rows")

Train: 7500 rows | Test: 2500 rows | Total: 10000 rows


## 6. Cleaning & Pre-processing
Equivalent to `recipe()` with `step_dummy()`, `step_normalize()`, `step_YeoJohnson()`.  
We use `ColumnTransformer` + `Pipeline` in scikit-learn.

In [11]:
num_cols = ["Age", "length_of_stay"]
cat_cols = ["Gender", "Blood Type", "Medical Condition",
            "Insurance Provider", "Admission Type", "Medication", "Test Results"]

# Numeric: YeoJohnson (handles skew) + StandardScaler
numeric_transformer = Pipeline(steps=[
    ("yeo",    PowerTransformer(method="yeo-johnson")),
    ("scaler", StandardScaler())
])

# Categorical: OneHotEncoder (equivalent to step_dummy)
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,    num_cols),
    ("cat", categorical_transformer, cat_cols)
])

print("Preprocessor defined.")

Preprocessor defined.


## 7. Model Specification & Hyperparameter Tuning
Equivalent to `linear_reg(penalty = tune(), mixture = 1)` (Lasso) + `tune_grid()` in tidymodels.

In [12]:
# Lasso pipeline (mixture=1 in tidymodels = Lasso in sklearn)
lasso_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model",        Lasso(max_iter=10000))
])

# Grid of penalty values (log-spaced, equivalent to grid_regular)
param_grid = {"model__alpha": np.logspace(-10, 1, 50)}

# Tune with 10-fold CV, scoring RMSE
grid_search = GridSearchCV(
    lasso_pipeline,
    param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Best alpha (penalty): {grid_search.best_params_['model__alpha']:.2e}")
print(f"Best CV RMSE:         {-grid_search.best_score_:.4f}")

0.02s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

Best alpha (penalty): 7.20e-03
Best CV RMSE:         0.8133


In [13]:
# Plot tuning results — equivalent to lasso_tune %>% autoplot()
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results["alpha"] = cv_results["param_model__alpha"].astype(float)
cv_results["rmse"]  = -cv_results["mean_test_score"]

fig = px.line(cv_results, x="alpha", y="rmse", log_x=True,
              title="Hyperparameter Tuning: RMSE vs Alpha (Penalty)",
              labels={"alpha": "Amount of Regularization (alpha)", "rmse": "RMSE (CV)"})
fig.add_vline(x=grid_search.best_params_["model__alpha"], line_dash="dash",
              annotation_text="Best alpha")
fig.show()

## 8. Fit with Best Hyperparameter
Equivalent to `finalize_workflow()` + `last_fit()`.

In [17]:
# Use the best estimator directly
best_model = grid_search.best_estimator_

# Evaluate on test set
y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
rsq  = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    ".metric":    ["rmse",  "mae",  "rsq"],
    ".estimate":  [rmse,    mae,    rsq]
})
print("Model Performance on Test Set")
metrics

Model Performance on Test Set


,.metric,.estimate
0,rmse,0.821065
1,mae,0.644168
2,rsq,0.001672


## 9. Prediction vs Truth
Equivalent to `collect_predictions() %>% ggplot(aes(log_price, .pred))`.

In [15]:
pred_df = pd.DataFrame({"Truth": y_test.values, "Prediction": y_pred})

fig = px.scatter(pred_df, x="Truth", y="Prediction",
                 title=f"Prediction vs Truth (log Billing Amount) — R²={rsq:.4f}, RMSE={rmse:.4f}",
                 opacity=0.4)

# Perfect prediction line (slope=1, intercept=0)
min_val, max_val = pred_df["Truth"].min(), pred_df["Truth"].max()
fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                         mode="lines", name="Perfect fit",
                         line=dict(color="black", dash="dash")))
fig.show()

## 10. Bonus — Compare Linear vs Ridge vs Lasso
Quick comparison of all three regularization approaches.

In [16]:
models = {
    "Linear Regression": Pipeline([("pre", preprocessor), ("model", LinearRegression())]),
    "Ridge (alpha=1)":   Pipeline([("pre", preprocessor), ("model", Ridge(alpha=1.0))]),
    "Lasso (best)": best_model
}

results = []
for name, model in models.items():
    if name != "Lasso (best)":
        model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        "Model": name,
        "RMSE":  round(np.sqrt(mean_squared_error(y_test, preds)), 4),
        "MAE":   round(mean_absolute_error(y_test, preds), 4),
        "R²":    round(r2_score(y_test, preds), 4)
    })

pd.DataFrame(results)

,Model,RMSE,MAE,R²
0,Linear Regression,0.8221,0.6438,-0.0010
1,Ridge (alpha=1),0.8217,0.6434,0.0000
2,Lasso (best),0.8211,0.6442,0.0017
